    GENERALIZED Error 
    SKEWED STUDENT

In [1]:
#VaR tests and vizualization
from scipy.stats import *
from scipy import stats 
from typing import Union, List, Literal, TypeAlias
import numpy as np
import numpy.typing as npt
import pandas as pd
from functools import wraps, partial
from arch import arch_model
import matplotlib.pyplot as plt

from Vares_simulations import *

from Vares import _terminal_returns, historical_var

Vares.ENABLE_TIMING = False

import pickle

#import the USD data 
with open('prices_usd.pkl', 'rb') as f: 
    prices_usd = pickle.load(f)

with open('original_returns_usd.pkl', 'rb') as f: 
    original_returns_usd = pickle.load(f)

with open('log_returns_usd.pkl', 'rb') as f: 
    log_returns_usd = pickle.load(f)    

original_returns_usd_scaled = original_returns_usd * 100
log_returns_usd_scaled = log_returns_usd * 100

from GARCH_VaR_delete_or_merge import fit_GARCH_VaR

import pickle

with open("norm_vars.pkl", "rb") as f:
    _norm_vars = pickle.load(f)

def turn_var_into_real_terms(var_array, prices):
    return [np.exp(log_delta) * price for log_delta, price in zip(var_array, prices)]

    GARCH(1,1)

In [ ]:
#allow for simulatoin of the GED process (?)

def _type_dist_simulation(dist_type: str, vol_type: str, **kwargs): 
    def simulate(returns, horizon=10, n_paths=50_000, alpha=0.01):
        model = arch_model(returns, vol=vol_type, p=1, q=1, dist=dist_type, **kwargs)
        # results = model.fit(disp='off', options={'maxiter': 4000, 'eps': 1e-8})
        results = model.fit(disp='off')
        forecast = results.forecast(horizon=horizon, method='simulation')
        mu = results.params.get("mu", 0)
        sigma2 = forecast.variance.values[0]
        sigma = sigma2 ** (1/2) #garch model estimates the variance of the underlyinh sample
        # nu = results.params.get('nu', 0)
        parameters = SimParams(volatility=sigma, mean=mu)

        # _ = solve_local_vol_gbm(parameters, 10, n_paths=n_paths)
        sim_returns = solve_local_vol_gbm_log(parameters, 10, n_paths=n_paths)

        # plt.hist(sim_returns, bins=50)
        # plt.show()

        VaR = historical_var(sim_returns, alpha)
        
        return VaR
    
    return simulate

In [4]:
GARCH_skewT_simulation = _type_dist_simulation('skewt', 'GARCH')
GARCH_GE_simulation = _type_dist_simulation('ged', 'GARCH')

In [5]:
garch_skewt_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_skewT_simulation) 
_garch_skewt_vars = np.multiply(garch_skewt_vars, -0.01)

garch_ge_vars = fit_GARCH_VaR(log_returns_usd_scaled, GARCH_GE_simulation) 
_garch_ge_vars = np.multiply(garch_ge_vars, -0.01)

In [7]:
import plotly.graph_objects as go

# Create figure
fig = go.Figure()

# Add the returns line (scaled)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y= prices_usd,
    mode='lines',
    name='Prices (RUB/USD)',
    line=dict(color='black', width=1),
    opacity=0.7
))

# Add GARCH(1,1)
fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_norm_vars, prices_usd),
    mode='lines',
    line=dict(color='red', width=0.5),
    name='GARCH(1, 1)'
))

fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_garch_skewt_vars, prices_usd),
    mode='lines',
    line=dict(color='orange', width=0.5),
    name='GARCH(1,1) (SkewT)'
))


fig.add_trace(go.Scatter(
    x=np.arange(len(prices_usd)),
    y=turn_var_into_real_terms(_garch_skewt_vars, prices_usd),
    mode='lines',
    line=dict(color='blue', width=0.5),
    name='GARCH(1,1) GED'
))

# fig.add_trace(go.Scatter(
#     x=np.arange(len(prices_usd)),
#     y=turn_var_into_real_terms(_t_egarch_vars, prices_usd),
#     mode='lines',
#     line=dict(color='green', width=0.5),
#     name='EGARCH(1,0,1) (S)'
# ))

# fig.add_trace(go.Scatter(
#     x=np.arange(len(prices_usd)),
#     y=turn_var_into_real_terms(_t_gjrgarjch_vars, prices_usd),
#     mode='lines',
#     line=dict(color='brown', width=0.5),
#     name='GJR-GARCH(1,1,1) (S)'
# ))

# Update layout
fig.update_layout(
    title=f'GARCH VaRs',
    width=1200,
    height=800,
    hovermode='x unified',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01
    )
)

fig.show()


    TARCH 
    FIGARCH
    EGARCH(1,0,1)
    EGARCH(1,1,1)
    EWMA + Student 
    RM2006 + Student 
    HARCH + Student 

In [ ]:
TARCH_skewT_simulation = _type_dist_simulation('skewt', 'GARCH', power=1.0)
TARCH_GE_simulation = _type_dist_simulation('ged', 'GARCH', power=1.0)

FIGARCH_skewT_simulation = _type_dist_simulation('skewt', 'FIGARCH', power=1.0)
FIGARCH_GE_simulation = _type_dist_simulation('ged', 'FIGARCH', power=1.0)

